# Identifying difficult to annotate tweets proof of concept

In [1]:
import pandas as pd

In the following dataset 20 contributors graded each tweet. 10 contributors graded the original sentiment evaluation.

In [2]:
df = pd.read_csv("weather-agg-DFE.csv")

Note the confidence score `what_emotion_does_the_author_express_specifically_about_the_weather:confidence` in the table below

In [3]:
df.head()

,_unit_id,_canary,_unit_state,_trusted_judgments,_last_judgment_at,what_emotion_does_the_author_express_specifically_about_the_weather,what_emotion_does_the_author_express_specifically_about_the_weather:confidence,gold_answer,tweet_id,tweet_text
0,314960380,NaN,finalized,20,8/24/13 0:21,Positive,0.8439,NaN,81990560,Grilling kabobs on the grill last night was am...
1,314960381,NaN,finalized,20,8/24/13 0:49,Negative,0.6963,NaN,84314377,The slowest day ever !! And the weather makes ...
2,314960382,NaN,finalized,20,8/24/13 0:55,Neutral / author is just sharing information,0.8802,NaN,82846118,Fire Weather Watch issued May 17 at 4:21PM CDT...
3,314960383,NaN,finalized,20,8/24/13 0:48,Positive,0.6897,NaN,82843785,Im going to lunch early today. The weather i...
4,314960384,NaN,finalized,20,8/24/13 1:19,Neutral / author is just sharing information,0.6153,NaN,82840144,Weekend Weather Causes Delays In I-270 Bridge ...


In [4]:
tweets = df[["what_emotion_does_the_author_express_specifically_about_the_weather:confidence","tweet_text"]]
tweets.columns= ['Confidence', 'text']
tweets

,Confidence,text
0,0.8439,Grilling kabobs on the grill last night was am...
1,0.6963,The slowest day ever !! And the weather makes ...
2,0.8802,Fire Weather Watch issued May 17 at 4:21PM CDT...
3,0.6897,Im going to lunch early today. The weather i...
4,0.6153,Weekend Weather Causes Delays In I-270 Bridge ...
...,...,...
995,0.8747,"good morning, it's sunny, pick up new car and ..."
996,0.7509,Just saw Snow White working at Lady Foot Locke...
997,0.5128,RT @mention: Do NOT go out to move your car in...
998,0.7472,Not outside but looking out the window at them...


Observe that there are 745 tweets that have a confidence larger tha 0.5

In [5]:
len(tweets[tweets['Confidence'] > 0.5])

745

In [6]:
tweets['Confidence'] = tweets['Confidence'] > 0.5

/var/folders/v4/cft4t7hn0_g2knsl5cd3c33m0000gp/T/ipykernel_93745/3106272242.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tweets['Confidence'] = tweets['Confidence'] > 0.5


In [7]:
tweets

,Confidence,text
0,True,Grilling kabobs on the grill last night was am...
1,True,The slowest day ever !! And the weather makes ...
2,True,Fire Weather Watch issued May 17 at 4:21PM CDT...
3,True,Im going to lunch early today. The weather i...
4,True,Weekend Weather Causes Delays In I-270 Bridge ...
...,...,...
995,True,"good morning, it's sunny, pick up new car and ..."
996,True,Just saw Snow White working at Lady Foot Locke...
997,True,RT @mention: Do NOT go out to move your car in...
998,True,Not outside but looking out the window at them...


## Simple model to find difficult to annotate tweets

#### Word embedding using universal-sentence-encoder-large

In [8]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text
from sklearn.model_selection import train_test_split

In [9]:
module_url = "https://tfhub.dev/google/universal-sentence-encoder-multilingual-large/3"
embedding = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return embedding(input)

2021-10-29 16:31:17.313433: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


module https://tfhub.dev/google/universal-sentence-encoder-multilingual-large/3 loaded


2021-10-29 16:31:21.639446: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(embed(tweets['text'].tolist()).numpy(), tweets['Confidence'], 
                                                    test_size=0.33, random_state=42)

In [11]:
# VOCAB_SIZE = 1000
# encoder = tf.keras.layers.TextVectorization(
#     max_tokens=VOCAB_SIZE)
# encoder.adapt(X_train)

In [12]:
# model = tf.keras.Sequential([
#     encoder,
#     tf.keras.layers.Embedding(
#         input_dim=len(encoder.get_vocabulary()),
#         output_dim=64,
#         # Use masking to handle the variable sequence lengths
#         mask_zero=True),
#     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
#     tf.keras.layers.Dense(64, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
# ])

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu',),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [14]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [19]:
model.fit(X_train, y_train, epochs=500, batch_size=20, validation_split=0.2)

Epoch 1/500
27/27 [==============================] - 0s 3ms/step - loss: 0.0172 - accuracy: 0.9907 - val_loss: 2.2260 - val_accuracy: 0.6567
Epoch 2/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0131 - accuracy: 0.9944 - val_loss: 2.1376 - val_accuracy: 0.6567
Epoch 3/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0124 - accuracy: 0.9963 - val_loss: 2.3648 - val_accuracy: 0.6791
Epoch 4/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0164 - accuracy: 0.9963 - val_loss: 2.1484 - val_accuracy: 0.6418
Epoch 5/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0051 - accuracy: 0.9981 - val_loss: 2.2351 - val_accuracy: 0.6791
Epoch 6/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0191 - accuracy: 0.9907 - val_loss: 2.1425 - val_accuracy: 0.6418
Epoch 7/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0088 - accuracy: 0.9963 - val_loss: 2.2708 - val_accuracy: 0.6866
Epoch 8/500
2

Epoch 59/500
27/27 [==============================] - 0s 2ms/step - loss: 5.7342e-04 - accuracy: 1.0000 - val_loss: 2.6722 - val_accuracy: 0.6642
Epoch 60/500
27/27 [==============================] - 0s 2ms/step - loss: 4.6713e-04 - accuracy: 1.0000 - val_loss: 2.6832 - val_accuracy: 0.6716
Epoch 61/500
27/27 [==============================] - 0s 2ms/step - loss: 4.7342e-04 - accuracy: 1.0000 - val_loss: 2.7130 - val_accuracy: 0.6716
Epoch 62/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0017 - accuracy: 1.0000 - val_loss: 2.7764 - val_accuracy: 0.6866
Epoch 63/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0035 - accuracy: 0.9981 - val_loss: 2.7924 - val_accuracy: 0.6716
Epoch 64/500
27/27 [==============================] - 0s 2ms/step - loss: 8.4718e-04 - accuracy: 1.0000 - val_loss: 2.7935 - val_accuracy: 0.7015
Epoch 65/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0097 - accuracy: 0.9981 - val_loss: 2.7953 - val_accurac

27/27 [==============================] - 0s 2ms/step - loss: 8.3618e-04 - accuracy: 1.0000 - val_loss: 3.4296 - val_accuracy: 0.6194
Epoch 117/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0020 - accuracy: 1.0000 - val_loss: 3.4389 - val_accuracy: 0.6567
Epoch 118/500
27/27 [==============================] - 0s 2ms/step - loss: 3.1187e-04 - accuracy: 1.0000 - val_loss: 3.4911 - val_accuracy: 0.6493
Epoch 119/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0021 - accuracy: 0.9981 - val_loss: 3.4079 - val_accuracy: 0.6343
Epoch 120/500
27/27 [==============================] - 0s 2ms/step - loss: 2.3536e-04 - accuracy: 1.0000 - val_loss: 3.4241 - val_accuracy: 0.6343
Epoch 121/500
27/27 [==============================] - 0s 2ms/step - loss: 2.1103e-04 - accuracy: 1.0000 - val_loss: 3.4461 - val_accuracy: 0.6418
Epoch 122/500
27/27 [==============================] - 0s 2ms/step - loss: 8.6672e-04 - accuracy: 1.0000 - val_loss: 3.5078 - val_accuracy: 

Epoch 173/500
27/27 [==============================] - 0s 2ms/step - loss: 7.7869e-04 - accuracy: 1.0000 - val_loss: 3.3001 - val_accuracy: 0.6716
Epoch 174/500
27/27 [==============================] - 0s 2ms/step - loss: 4.4869e-04 - accuracy: 1.0000 - val_loss: 3.3800 - val_accuracy: 0.6716
Epoch 175/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0020 - accuracy: 0.9981 - val_loss: 3.3026 - val_accuracy: 0.6642
Epoch 176/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0016 - accuracy: 1.0000 - val_loss: 3.4366 - val_accuracy: 0.6791
Epoch 177/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0017 - accuracy: 1.0000 - val_loss: 3.4623 - val_accuracy: 0.6642
Epoch 178/500
27/27 [==============================] - 0s 2ms/step - loss: 9.3034e-05 - accuracy: 1.0000 - val_loss: 3.4188 - val_accuracy: 0.6493
Epoch 179/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0011 - accuracy: 1.0000 - val_loss: 3.3890 - val_accu

27/27 [==============================] - 0s 2ms/step - loss: 3.1335e-04 - accuracy: 1.0000 - val_loss: 2.8593 - val_accuracy: 0.6716
Epoch 230/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0011 - accuracy: 1.0000 - val_loss: 2.8972 - val_accuracy: 0.6716
Epoch 231/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0012 - accuracy: 1.0000 - val_loss: 2.8309 - val_accuracy: 0.6343
Epoch 232/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0013 - accuracy: 1.0000 - val_loss: 2.9604 - val_accuracy: 0.6567
Epoch 233/500
27/27 [==============================] - 0s 2ms/step - loss: 3.3435e-04 - accuracy: 1.0000 - val_loss: 3.0330 - val_accuracy: 0.6567
Epoch 234/500
27/27 [==============================] - 0s 2ms/step - loss: 3.1882e-04 - accuracy: 1.0000 - val_loss: 3.0599 - val_accuracy: 0.6567
Epoch 235/500
27/27 [==============================] - 0s 2ms/step - loss: 2.8627e-04 - accuracy: 1.0000 - val_loss: 3.0773 - val_accuracy: 0.65

27/27 [==============================] - 0s 2ms/step - loss: 5.7427e-04 - accuracy: 1.0000 - val_loss: 2.8937 - val_accuracy: 0.6866
Epoch 286/500
27/27 [==============================] - 0s 2ms/step - loss: 2.8704e-04 - accuracy: 1.0000 - val_loss: 2.9033 - val_accuracy: 0.6716
Epoch 287/500
27/27 [==============================] - 0s 2ms/step - loss: 4.7220e-04 - accuracy: 1.0000 - val_loss: 2.9259 - val_accuracy: 0.6716
Epoch 288/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0035 - accuracy: 0.9981 - val_loss: 2.9909 - val_accuracy: 0.7164
Epoch 289/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0012 - accuracy: 1.0000 - val_loss: 2.9673 - val_accuracy: 0.6716
Epoch 290/500
27/27 [==============================] - 0s 2ms/step - loss: 7.2120e-04 - accuracy: 1.0000 - val_loss: 2.9893 - val_accuracy: 0.6716
Epoch 291/500
27/27 [==============================] - 0s 2ms/step - loss: 8.0774e-04 - accuracy: 1.0000 - val_loss: 2.9664 - val_accuracy: 

27/27 [==============================] - 0s 2ms/step - loss: 0.0051 - accuracy: 0.9981 - val_loss: 3.8360 - val_accuracy: 0.6493
Epoch 342/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0017 - accuracy: 1.0000 - val_loss: 3.7592 - val_accuracy: 0.6343
Epoch 343/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0207 - accuracy: 0.9944 - val_loss: 3.4199 - val_accuracy: 0.6716
Epoch 344/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0123 - accuracy: 0.9907 - val_loss: 3.2653 - val_accuracy: 0.7164
Epoch 345/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0439 - accuracy: 0.9944 - val_loss: 2.7982 - val_accuracy: 0.6567
Epoch 346/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0074 - accuracy: 0.9981 - val_loss: 2.8584 - val_accuracy: 0.6716
Epoch 347/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0225 - accuracy: 0.9925 - val_loss: 2.5526 - val_accuracy: 0.6567
Epoch 348/500

27/27 [==============================] - 0s 2ms/step - loss: 0.0020 - accuracy: 1.0000 - val_loss: 3.5587 - val_accuracy: 0.7090
Epoch 398/500
27/27 [==============================] - 0s 2ms/step - loss: 6.3613e-04 - accuracy: 1.0000 - val_loss: 3.5628 - val_accuracy: 0.7015
Epoch 399/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0011 - accuracy: 1.0000 - val_loss: 3.5170 - val_accuracy: 0.6716
Epoch 400/500
27/27 [==============================] - 0s 2ms/step - loss: 1.1294e-04 - accuracy: 1.0000 - val_loss: 3.5170 - val_accuracy: 0.6642
Epoch 401/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0053 - accuracy: 0.9981 - val_loss: 3.6627 - val_accuracy: 0.7164
Epoch 402/500
27/27 [==============================] - 0s 2ms/step - loss: 5.5336e-04 - accuracy: 1.0000 - val_loss: 3.6564 - val_accuracy: 0.7090
Epoch 403/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0050 - accuracy: 0.9963 - val_loss: 3.5534 - val_accuracy: 0.6716
E

Epoch 454/500
27/27 [==============================] - 0s 2ms/step - loss: 6.3846e-04 - accuracy: 1.0000 - val_loss: 3.5654 - val_accuracy: 0.6418
Epoch 455/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0064 - accuracy: 0.9981 - val_loss: 3.5777 - val_accuracy: 0.6642
Epoch 456/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0094 - accuracy: 0.9981 - val_loss: 3.2784 - val_accuracy: 0.6343
Epoch 457/500
27/27 [==============================] - 0s 2ms/step - loss: 2.7471e-04 - accuracy: 1.0000 - val_loss: 3.2464 - val_accuracy: 0.6269
Epoch 458/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0052 - accuracy: 0.9963 - val_loss: 3.4423 - val_accuracy: 0.7239
Epoch 459/500
27/27 [==============================] - 0s 2ms/step - loss: 0.0032 - accuracy: 0.9981 - val_loss: 3.1959 - val_accuracy: 0.6642
Epoch 460/500
27/27 [==============================] - 0s 2ms/step - loss: 1.9301e-04 - accuracy: 1.0000 - val_loss: 3.2029 - val_accu

In [20]:
test_loss, test_acc = model.evaluate(X_test, y_test)

11/11 [==============================] - 0s 1ms/step - loss: 3.9857 - accuracy: 0.7515


In [21]:
def is_difficult_to_annotate(tweet:str) -> bool:
    answer = model.predict(embed([tweet]).numpy())
    return answer[0][0] < 0.5

In [24]:
is_difficult_to_annotate("RT @TeamCavuto: Protesters stage #DieIn protests in @Apple store in NYC... Is it me, or is this anger misplaced? RETWEET if you agree.")


False